# Notebook 12 — SciPy: ODEs and Numerical Integration

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 12 of 20  
**Prerequisites:** Notebook 11  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Biological systems evolve over time: gene expression changes in response to a signal,
a population grows and competes for resources, a virus spreads through a host.
These dynamics are described by ordinary differential equations (ODEs). Numerical
integration solves ODEs when closed-form solutions don't exist — which is almost always.

---
## Step 2 — Intuition

An ODE says: 'the rate of change of a quantity depends on its current value.' A
numerical solver takes small time steps, evaluating the rate at each step and using
it to predict the next value. Step size matters: too large = inaccurate; too small =
too slow. Adaptive solvers (like RK45) adjust step size automatically.

---
## Step 3 — Biological Background

**SIR epidemic model:**

A compartment model for disease spread. The population is divided into:
- $S$: Susceptible (can be infected)
- $I$: Infected (and infectious)
- $R$: Recovered (and immune)

$$\frac{dS}{dt} = -\beta S I / N$$
$$\frac{dI}{dt} = \beta S I / N - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

where $\beta$ = transmission rate, $\gamma$ = recovery rate, $N$ = population.
This is the foundational model in epidemiology — directly relevant to Module 15
(agent-based modelling).

---
## Step 4 — Mathematical Explanation

**Basic reproduction number:** $R_0 = \beta / \gamma$

- $R_0 > 1$: epidemic grows ($I$ increases initially)
- $R_0 < 1$: epidemic dies out ($I$ decreases from the start)
- $R_0 = 1$: threshold

**Herd immunity threshold:** $1 - 1/R_0$ — the fraction of the population that must
be immune for an epidemic not to grow.

**Runge-Kutta 4 (RK4) step:**
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

where $k_1, k_2, k_3, k_4$ are derivatives evaluated at intermediate points.

---
## Step 5 — Computational Explanation

| API | Use case | Notes |
|-----|----------|-------|
| `scipy.integrate.solve_ivp` | Solve IVP (initial value problem) | RK45 default; adaptive step |
| `scipy.integrate.odeint` | Older API; LSODA solver | Still common in legacy code |
| `scipy.integrate.quad` | Definite integral $\int_a^b f(x) dx$ | High accuracy quadrature |

For stiff ODEs (fast and slow timescales), use `method='Radau'` or `method='BDF'` in
`solve_ivp`. Gene regulatory networks are often stiff.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp, quad
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — SIR model
def sir_odes(t: float, y: np.ndarray, beta: float, gamma: float, N: float) -> list:
    """SIR ODE system. Returns [dS/dt, dI/dt, dR/dt]."""
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

# Parameters
N = 10_000        # population
beta = 0.3        # transmission rate (contacts per day × prob. transmission)
gamma = 0.1       # recovery rate (1/gamma = mean infectious period in days)
R0 = beta / gamma
print(f"R0 = {R0:.1f}")
print(f"Herd immunity threshold = {1 - 1/R0:.1%}")

# Initial conditions: 1 infected, rest susceptible
S0, I0, R0_comp = N - 1, 1, 0
y0 = [S0, I0, R0_comp]

# Solve over 160 days
sol = solve_ivp(
    fun=sir_odes,
    t_span=(0, 160),
    y0=y0,
    args=(beta, gamma, N),
    method="RK45",
    dense_output=True,
    rtol=1e-6,
    atol=1e-8,
)

print(f"Solver success: {sol.success}")
print(f"Time steps taken: {len(sol.t)}")

In [ ]:
# Cell 6.2 — Gene regulatory network: simple negative feedback
# mRNA (m) and protein (p) with negative feedback
# dm/dt = alpha_m / (1 + (p/K)^n) - delta_m * m
# dp/dt = alpha_p * m - delta_p * p

def grn_feedback(t, y, alpha_m, alpha_p, delta_m, delta_p, K, n):
    m, p = y
    dm = alpha_m / (1 + (p/K)**n) - delta_m * m
    dp = alpha_p * m - delta_p * p
    return [dm, dp]

params = dict(alpha_m=10.0, alpha_p=2.0, delta_m=0.5, delta_p=0.1, K=50.0, n=2)
grn_sol = solve_ivp(
    grn_feedback, t_span=(0, 100), y0=[0.0, 0.0],
    args=tuple(params.values()), method="RK45", dense_output=True,
)

t_eval = np.linspace(0, 100, 500)
m_vals, p_vals = grn_sol.sol(t_eval)
print(f"Steady-state mRNA:    {m_vals[-1]:.2f}")
print(f"Steady-state protein: {p_vals[-1]:.2f}")

In [ ]:
# Cell 6.3 — Numerical integration: area under a distribution
from scipy.stats import norm

# Probability that expression is between 4 and 7 under N(5, 1.5^2)
prob, err = quad(norm.pdf, 4, 7, args=(5.0, 1.5))
print(f"P(4 < X < 7) = {prob:.4f}  (error estimate: {err:.2e})")
# Cross-check with scipy.stats CDF
prob_exact = norm.cdf(7, 5, 1.5) - norm.cdf(4, 5, 1.5)
print(f"Exact CDF:     {prob_exact:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — SIR dynamics
t_plot = np.linspace(0, 160, 500)
S_plot, I_plot, R_plot = sol.sol(t_plot)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# SIR time series
ax = axes[0]
ax.plot(t_plot, S_plot/N, label="Susceptible", color="steelblue")
ax.plot(t_plot, I_plot/N, label="Infected",    color="tomato")
ax.plot(t_plot, R_plot/N, label="Recovered",   color="seagreen")
ax.set_xlabel("Days"); ax.set_ylabel("Fraction of population")
ax.set_title(f"SIR model (R₀ = {R0:.1f})")
ax.legend(); ax.set_ylim(0, 1)

# GRN dynamics
ax = axes[1]
ax.plot(t_eval, m_vals, label="mRNA", color="steelblue")
ax.plot(t_eval, p_vals, label="Protein", color="tomato")
ax.set_xlabel("Time"); ax.set_ylabel("Concentration")
ax.set_title("Gene regulatory network — negative feedback")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Modify the SIR model to become an SEIR model (add Exposed compartment with rate $\sigma$).
   Simulate with $\sigma = 0.2$.
2. For the GRN model, vary the Hill coefficient $n$ from 1 to 4. How does the
   steepness of the repression curve change the dynamics?
3. What happens to the SIR epidemic when $R_0 < 1$? Simulate with $\beta = 0.05$.
4. Write a `simulate_sir(beta, gamma, N, t_max, I0)` function in `compbio_utils/stats.py`
   that returns a DataFrame with columns `t`, `S`, `I`, `R`.

---
## Quiz — Active Recall

1. What is $R_0$ and what does $R_0 = 3$ mean for a disease?
2. In the SIR model, when is $dI/dt = 0$ (the epidemic peak)?
3. What does `dense_output=True` enable in `solve_ivp`?
4. What is a stiff ODE and why does it require a different solver?
5. How would you numerically verify that $S + I + R = N$ is preserved throughout
   the simulation?

---
## Reflection

**Date completed:** ____________________

> *[Could you write the SIR ODE system from memory? Is simulate_sir in compbio_utils? What was hardest about the SEIR extension?]*

---
*Next: `13_reusable_scientific_code.ipynb`*